# 🏆 Credit Risk & Loan Default Prediction: 04 - Hyperparameter Tuning & Final Pipeline Calibration

### Objective:
- Optimize CatBoost hyperparameters with RandomizedSearchCV
- Generate Out-Of-Fold (OOF) predicted probabilities
- Calibrate optimal risk decision threshold (Precision-Recall / F-Beta analysis)
- Finalize, evaluate on held-out test set, and serialize production pipeline artifact


In [ ]:
import numpy as np
import pandas as pd
import joblib
import json
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_curve,
    classification_report, confusion_matrix, precision_score, recall_score, f1_score, fbeta_score
)
from catboost import CatBoostClassifier

import sys
sys.path.append('..')
from src.features.feature_engineering import FeatureEngineer, CatBoostPreprocessor, get_feature_metadata


## 1. Load Data & Prepare Partitions


In [ ]:
df = pd.read_csv('../data/raw/application_train.csv')
X = df.drop(columns='TARGET')
y = df['TARGET']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Train samples: {len(X_train):,}, Test samples: {len(X_test):,}')

metadata = get_feature_metadata(X_train)
cols_to_drop = metadata['cols_to_drop']
categorical_numeric = metadata['categorical_numeric']
categorical_features = metadata['categorical_features']


## 2. Final Optimized Pipeline Architecture


In [ ]:
final_model = CatBoostClassifier(
    depth=7,
    eval_metric='AUC',
    iterations=700,
    l2_leaf_reg=5,
    learning_rate=0.05,
    loss_function='Logloss',
    random_seed=42,
    random_strength=0,
    verbose=0
)

final_pipeline = Pipeline([
    ('feature_engineering', FeatureEngineer(
        cols_to_drop=cols_to_drop,
        categorical_numeric=categorical_numeric
    )),
    ('preprocessor', CatBoostPreprocessor(
        categorical_features=categorical_features
    )),
    ('model', final_model)
])


## 3. Out-Of-Fold (OOF) Prediction for Threshold Optimization


In [ ]:
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)

OOF = cross_val_predict(
    final_pipeline,
    X_train,
    y_train,
    cv=cv,
    method='predict_proba',
    n_jobs=-1,
    params={'model__cat_features': categorical_features}
)

y_oof_prob = OOF[:, 1]
print(f'OOF ROC-AUC Score: {roc_auc_score(y_train, y_oof_prob):.4f}')
print(f'OOF PR-AUC Score:  {average_precision_score(y_train, y_oof_prob):.4f}')


## 4. Precision-Recall & F-Beta Threshold Calibration
For credit default detection, missing a defaulting borrower is far more costly than conducting manual review. Hence we optimize threshold for high recall (F-2 score / recall >= 0.75).


In [ ]:
thresholds = np.arange(0.01, 0.50, 0.01)
metrics = []

for t in thresholds:
    y_pred = (y_oof_prob >= t).astype(int)
    metrics.append({
        'threshold': round(t, 2),
        'precision': precision_score(y_train, y_pred, zero_division=0),
        'recall': recall_score(y_train, y_pred, zero_division=0),
        'f1': f1_score(y_train, y_pred, zero_division=0),
        'f2': fbeta_score(y_train, y_pred, beta=2, zero_division=0)
    })

threshold_df = pd.DataFrame(metrics)
print('Optimal Threshold Options with Recall >= 0.75:')
print(threshold_df[threshold_df['recall'] >= 0.75].sort_values(by='precision', ascending=False).head(10))

FINAL_THRESHOLD = 0.05
print(f'\nSelected Business Decision Threshold: {FINAL_THRESHOLD}')


## 5. Train Final Production Pipeline on Full Train Set


In [ ]:
final_pipeline.fit(
    X_train,
    y_train,
    model__cat_features=categorical_features
)

test_probs = final_pipeline.predict_proba(X_test)[:, 1]
test_preds = (test_probs >= FINAL_THRESHOLD).astype(int)

print(f'Test ROC-AUC: {roc_auc_score(y_test, test_probs):.4f}')
print(f'Test PR-AUC:  {average_precision_score(y_test, test_probs):.4f}')
print('\nTest Classification Report at threshold = 0.05:')
print(classification_report(y_test, test_preds))


## 6. Serialize Production Artifacts & Metadata


In [ ]:
import os

os.makedirs('../artifacts', exist_ok=True)
joblib.dump(final_pipeline, '../artifacts/model_pipeline.joblib')

meta = {
    'model_name': 'CatBoost Credit Risk & Default Predictor',
    'version': '1.0.0',
    'algorithm': 'CatBoostClassifier',
    'optimal_threshold': FINAL_THRESHOLD,
    'metrics': {
        'roc_auc': float(round(roc_auc_score(y_test, test_probs), 4)),
        'pr_auc': float(round(average_precision_score(y_test, test_probs), 4)),
        'recall_at_optimal_threshold': float(round(recall_score(y_test, test_preds), 4)),
        'precision_at_optimal_threshold': float(round(precision_score(y_test, test_preds), 4)),
        'f1_at_optimal_threshold': float(round(f1_score(y_test, test_preds), 4))
    },
    'categorical_features': categorical_features,
    'categorical_numeric': categorical_numeric,
    'cols_to_drop': cols_to_drop
}

with open('../artifacts/model_metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('Model and metadata successfully serialized to ../artifacts/')
